# W2D1 — Understand the Data — Lab

**Week 2 · Day 1 · Data Engineering** · Lab

A shop's order system has been exporting to CSV for eighteen months, and nobody has checked the
export. Today you open it for the first time. You will not clean it and you will not model it —
you will find out **what is in it and what is wrong with it**, because every decision you make for
the rest of the week depends on what you learn in the next 115 minutes. The question behind the
whole week is whether an order will be **returned**, and by the end of today you will know how
often that happens and roughly what the data will let you say about it. You leave with
`data_quality_notes.md`, the list of problems D3 has to fix.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ١ — فهم البيانات

**الأسبوع ٢ · اليوم ١ · هندسة البيانات** · معمل عملي

يُصدِّر نظام طلبات أحد المتاجر ملفات CSV منذ ثمانية عشر شهرًا، ولم يفحص أحد هذا التصدير. واليوم
تفتحه لأول مرة. لن تنظّف البيانات ولن تبني نموذجًا — بل ستكتشف **ما الذي فيها وما الخطأ فيها**،
لأن كل قرار تتّخذه في بقية الأسبوع يعتمد على ما تتعلّمه في المئة وخمس عشرة دقيقة القادمة. والسؤال
الذي يقف خلف الأسبوع كله هو: هل سيُرجَع الطلب (`returned`)؟ وبنهاية اليوم ستعرف كم مرة يحدث ذلك،
وما الذي تسمح لك البيانات بقوله عنه تقريبًا. وستخرج بملف `data_quality_notes.md`، وهو قائمة
المشكلات التي على معمل اليوم الثالث إصلاحها.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Describe a dataset's shape, columns and dtypes, and say which columns are numeric and which are categorical.
- Measure missing values, duplicate rows and inconsistent category spellings, and say how much of each there is.
- Recognise that one text column holds several date formats, and explain why that breaks naive parsing.
- Find values that are impossible rather than merely unusual, and argue why they are impossible.
- State the class balance of the `returned` target and explain why a 23% positive rate matters.
- Turn a question about the data into a small `groupby` investigation and interpret the answer.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- وصف شكل مجموعة البيانات وأعمدتها وأنواعها، وتحديد الأعمدة الرقمية والأعمدة الفئوية.
- قياس القيم المفقودة والصفوف المكرّرة وتضارب تهجئة الفئات، وتحديد مقدار كل منها.
- إدراك أن عمودًا نصيًّا واحدًا يحمل عدة صيغ للتاريخ، وشرح سبب إفساد ذلك للتحليل الساذج.
- إيجاد القيم المستحيلة لا الغريبة فحسب، وتبرير استحالتها.
- ذكر توازن الفئات في هدف `returned` وشرح أهمية نسبة إيجابية تبلغ ٢٣٪.
- تحويل سؤال عن البيانات إلى تحقيق صغير باستخدام `groupby` وتفسير نتيجته.

</div>

## About the data

**Dataset:** `messy_sales` — built for this course · CC0 · 5,000 rows

One row is one sales order: when it was placed and shipped, which city and channel it came through,
the customer's loyalty tier, how many units at what price, and what the shop eventually recognised
as revenue. The column the week is about is **`returned`** — 1 if the customer sent the order back,
0 if they kept it. Predicting returns is worth doing because a return costs the shop the shipping
both ways plus the handling, so knowing which orders are at risk changes how you pack, price and
promise them.

**Watch out:** the whole file is the gotcha. It was built broken on purpose — missing values in
three different patterns, four date formats in one column, duplicated rows, fourteen spellings of
five cities, and **two columns that leak a target**. You are not expected to find all of that
today. You are expected to find *some* of it, and to write down what you find.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `messy_sales` — أُعدّت لهذه الدورة · رخصة CC0 · ٥٠٠٠ صف

الصف الواحد هو طلب بيع واحد: متى قُدّم ومتى شُحن، ومن أي مدينة وقناة جاء، ومستوى ولاء العميل، وكم
وحدة بأي سعر، وما الإيراد الذي اعترف به المتجر في النهاية. والعمود الذي يدور حوله الأسبوع هو
**`returned`** — ويساوي ١ إذا أعاد العميل الطلب و٠ إذا احتفظ به. والتنبّؤ بالإرجاع يستحق العناء
لأن الإرجاع يكلّف المتجر الشحن في الاتجاهين إضافة إلى المناولة، فمعرفة الطلبات المعرّضة للخطر
تغيّر طريقة التغليف والتسعير والوعود.

**انتبه:** الملف كله مشكلة. صُمّم معطوبًا عن قصد — قيم مفقودة بثلاثة أنماط مختلفة، وأربع صيغ
للتاريخ في عمود واحد، وصفوف مكرّرة، وأربع عشرة تهجئة لخمس مدن، و**عمودان يُسرّبان الهدف**. ولا
يُتوقّع منك أن تجد كل ذلك اليوم، بل أن تجد **بعضه** وأن تدوّن ما وجدت.

</div>

## Setup

Run the cell below first. It installs anything missing, fixes the random seed, and finds the
dataset — whether you are on your own machine or on Google Colab.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُثبّت ما ينقص، وتُثبّت البذرة العشوائية، وتجد ملف البيانات — سواء كنت
على جهازك أو على Google Colab.

</div>

In [1]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("pandas")                         # this lab is pandas only — no modelling today
seed_everything(42)                      # course-wide seed

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

DATA = get_dataset("messy_sales")
print(describe_dataset("messy_sales"))
print("\n", versions(), "| device:", device())

⚠️  Checksum mismatch for messy_sales.csv
  expected: fc71cf3e578ac8a93c823491b70487f461857eefd5ec50348f487677aedd23fb
  actual:   a75cb5f8a8ef10446e07b33eec2df00b2afb12f2642483d57b842b85628d09ae
The file may be corrupt or the registry may be out of date.
📁 messy_sales: using cached file at C:\AIEP_Olo_student\shared\data_cache\messy_sales.csv
messy_sales  (5000 rows, 0.41 MB)
  Source:  Built for this course
  Licence: CC0 (classroom fixture, generated)
  Target:  returned

  EN: Deliberately broken data: missing values in three different patterns, dates in four formats, 120 duplicated rows, fourteen spellings of five cities, and two columns that leak a target. Built so every cleaning technique in W2D3 has something real to fix, and so one dataset carries the whole Week 2 story from data quality through EDA to imbalanced classification. `returned` is 23% positive and genuinely hard — honest features reach AUC ~0.66, while the leaked column reaches ~0.999. Generated by tools/make_datas

## Section 1 — Warm-up  (≈25 min)

Guided. The code below works. Run it, read it, then change the one parameter each task points at
and observe what moves. Nothing here is a puzzle — the point is to get the file open and to
remember the pandas verbs you will need all week.

<div dir="rtl" align="right">

## القسم الأول — التهيئة (نحو ٢٥ دقيقة)

قسم موجَّه. الشيفرة أدناه تعمل. شغّلها واقرأها، ثم غيّر المعامل الذي تشير إليه كل مهمة ولاحظ ما
يتغيّر. لا يوجد لغز هنا — الهدف هو فتح الملف واستعادة أفعال pandas التي ستحتاجها طوال الأسبوع.

</div>

In [2]:
# Working code. Nothing to fill in here.
# Note we read with pandas' defaults — no parse_dates, no dtype hints. Seeing what
# pandas guesses on its own is the first piece of evidence about the file.
raw = pd.read_csv(DATA)

print(f"rows: {len(raw):,}   columns: {raw.shape[1]}")
raw.head()

rows: 5,000   columns: 13


,order_id,order_date,ship_date,city,channel,customer_tier,quantity,unit_price,discount_pct,commission_paid,refund_amount,revenue,returned
0,102389,2025-06-12,2025-06-21,DAMMAM,online,silver,32,53.17,0.244,38.32,0.00,1277.35,0
1,102810,15 Sep 2024,2024-09-18,Riyadh,online,bronze,26,112.91,0.198,72.12,1177.01,2403.99,1
2,101306,2025/04/24,2025-05-08,TABUK,online,bronze,27,33.67,0.192,21.52,0.00,717.36,0
3,104508,2024-04-15,2024-04-16,riyadh,store,gold,9,14.78,0.293,2.83,0.00,94.25,0
4,102377,18 Feb 2025,2025-03-02,Abha,online,silver,13,16.49,0.061,5.98,0.00,199.28,0


### Task 1.1 — Change what you look at

`head()` shows the first five rows, which is the least representative sample in the file — early
rows are often the cleanest. Change `n` below, and swap `.head` for `.sample`. Keep `random_state`
fixed so your neighbour sees the same rows you do.

**Look for:** a column whose values do not all look like they were written by the same system.

<div dir="rtl" align="right">

### المهمة ١٫١ — غيّر ما تنظر إليه

تُظهر `head()` أول خمسة صفوف، وهي أقل عيّنة تمثيلًا في الملف — فالصفوف الأولى غالبًا أنظفها. غيّر
قيمة `n` أدناه، واستبدل `.head` بـ `.sample`، مع تثبيت `random_state` كي يرى زميلك الصفوف نفسها.

**ابحث عن:** عمود لا تبدو قيمه كلها مكتوبة بالنظام نفسه.

</div>

In [3]:
# Change n, then switch .head(n) to .sample(n, random_state=0) and re-run.
n = 5

raw.head(n)

,order_id,order_date,ship_date,city,channel,customer_tier,quantity,unit_price,discount_pct,commission_paid,refund_amount,revenue,returned
0,102389,2025-06-12,2025-06-21,DAMMAM,online,silver,32,53.17,0.244,38.32,0.00,1277.35,0
1,102810,15 Sep 2024,2024-09-18,Riyadh,online,bronze,26,112.91,0.198,72.12,1177.01,2403.99,1
2,101306,2025/04/24,2025-05-08,TABUK,online,bronze,27,33.67,0.192,21.52,0.00,717.36,0
3,104508,2024-04-15,2024-04-16,riyadh,store,gold,9,14.78,0.293,2.83,0.00,94.25,0
4,102377,18 Feb 2025,2025-03-02,Abha,online,silver,13,16.49,0.061,5.98,0.00,199.28,0


### Task 1.2 — Ask what pandas decided

`dtypes` is pandas' guess, not the truth. A column of dates that pandas calls `object` is a column
of *text* that happens to contain dates — and text does not sort, subtract or compare like a date.

**Look for:** which columns came back as `object`, and ask yourself for each one whether that is
what you would have chosen.

<div dir="rtl" align="right">

### المهمة ١٫٢ — اسأل عمّا قرّره pandas

`dtypes` هو تخمين pandas لا الحقيقة. فعمود تواريخ يسمّيه pandas `object` هو عمود **نص** يصادف أنه
يحتوي تواريخ — والنص لا يُرتَّب ولا يُطرح ولا يُقارن مثل التاريخ.

**ابحث عن:** الأعمدة التي عادت بنوع `object`، واسأل نفسك عن كل منها: هل هذا ما كنت ستختاره؟

</div>

In [4]:
# Working code. Read the two lists it prints.
numeric_cols = raw.select_dtypes(include="number").columns.tolist()
object_cols = raw.select_dtypes(include="object").columns.tolist()

print("pandas thinks these are numeric: ", numeric_cols)
print("pandas thinks these are text:    ", object_cols)
print()
raw.dtypes.to_frame("dtype")

pandas thinks these are numeric:  ['order_id', 'quantity', 'unit_price', 'discount_pct', 'commission_paid', 'refund_amount', 'revenue', 'returned']
pandas thinks these are text:     ['order_date', 'ship_date', 'city', 'channel', 'customer_tier']



C:\Users\pc\AppData\Local\Temp\ipykernel_6988\948717021.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = raw.select_dtypes(include="object").columns.tolist()


,dtype
order_id,int64
order_date,str
ship_date,str
city,str
channel,str
customer_tier,str
quantity,int64
unit_price,float64
discount_pct,float64
commission_paid,float64


## Section 2 — Core  (≈60 min)

This is the lab. Each task has a goal; you write the code. You are building evidence, not fixing
anything — resist the urge to clean as you go. D3 is the cleaning lab, and it will go faster if
today's notes are precise.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي (نحو ٦٠ دقيقة)

هذا هو صلب المعمل. لكل مهمة هدف، وأنت من يكتب الشيفرة. أنت تجمع أدلّة ولا تصلح شيئًا — قاوم
الرغبة في التنظيف أثناء العمل. فمعمل اليوم الثالث هو معمل التنظيف، وسيكون أسرع إذا كانت ملاحظات
اليوم دقيقة.

</div>

### Task 2.1 — Measure what is missing

A count of nulls is not useful on its own; a *rate* is. Produce one table with, for each column,
the number of missing values and the share of rows they represent, sorted worst first, showing
only the columns that actually have some.

**Then answer, in the markdown cell after:** one column is far worse than the others. Is that
column missing because nobody filled it in, or is something about *which* rows go missing?

<div dir="rtl" align="right">

### المهمة ٢٫١ — قِس ما هو مفقود

عدد القيم الفارغة وحده غير مفيد، بل **النسبة** هي المفيدة. أنتج جدولًا واحدًا يبيّن لكل عمود عدد
القيم المفقودة وحصتها من الصفوف، مرتّبًا من الأسوأ، ومقتصرًا على الأعمدة التي فيها نقص فعلًا.

**ثم أجب في خلية Markdown التالية:** هناك عمود أسوأ من البقية بكثير. هل هو مفقود لأن أحدًا لم
يملأه، أم أن هناك شيئًا مشتركًا بين **الصفوف** التي تغيب فيها القيمة؟

</div>

In [5]:
missing_count = raw.isna().sum()
missing_rate = missing_count / len(raw) * 100

missing = pd.DataFrame({"missing": missing_count, "rate": missing_rate})
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)

print(missing.to_string(formatters={"rate": "{:.1f}%".format}))

              missing  rate
discount_pct     1473 29.5%


In [6]:
by_channel = raw.groupby("channel")["discount_pct"].apply(lambda x: x.isna().mean())

print("discount_pct missing rate by channel:")
print(by_channel.to_string(float_format="{:.1%}".format))

discount_pct missing rate by channel:
channel
online    17.5%
phone    100.0%
store     17.7%


**Write your answer here.** Is `discount_pct` missing at random? What did the breakdown by
`channel` tell you, and what would go wrong if you filled every gap with the column's average?

*(Replace this sentence with two or three of your own.)*

<div dir="rtl" align="right">

**اكتب إجابتك هنا.** هل `discount_pct` مفقود عشوائيًّا؟ ماذا أخبرك التوزيع حسب `channel`، وما الذي
سيحدث لو ملأت كل فجوة بمتوسّط العمود؟

*(استبدل هذه الجملة بجملتين أو ثلاث من عندك.)*

</div>

### Task 2.2 — Count the duplicates, then ask whether they matter

Find how many rows are exact duplicates of another row. Then — and this is the part people skip —
check whether removing them would move the data. Compare the mean of `revenue` and the rate of
`returned` before and after dropping them.

A duplicate set that changes the target rate is a different problem from one that does not.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — عُدّ المكرّرات ثم اسأل هل تهمّ

جِد كم صفًّا يكرّر صفًّا آخر تمامًا. ثم — وهذا ما يتخطّاه الناس — تحقّق ممّا إذا كان حذفها سيحرّك
البيانات. قارن متوسّط `revenue` ونسبة `returned` قبل الحذف وبعده.

فمجموعة مكرّرات تغيّر نسبة الهدف مشكلة مختلفة عن مجموعة لا تغيّرها.

</div>

In [7]:
n_duplicates = raw.duplicated().sum()
dedup = raw.drop_duplicates()

comparison = pd.DataFrame({
    "rows": [len(raw), len(dedup)],
    "mean_revenue": [raw["revenue"].mean(), dedup["revenue"].mean()],
    "mean_returned": [raw["returned"].mean(), dedup["returned"].mean()],
}, index=["with_duplicates", "deduplicated"])

print(f"{n_duplicates} exact duplicate rows\n")
print(comparison.to_string(float_format="{:.4f}".format))

120 exact duplicate rows

                 rows  mean_revenue  mean_returned
with_duplicates  5000      823.4627         0.2308
deduplicated     4880      823.0218         0.2299


### Task 2.3 — Find the inconsistent categories

Three columns are categorical: `city`, `channel` and `customer_tier`. For each, list the distinct
values and how often each occurs.

Two of the three are clean. One is not — and the damage is invisible until you look at the exact
strings. Count how many distinct values `city` has, and how many actual cities you think that is.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — جِد الفئات غير المتّسقة

ثلاثة أعمدة فئوية: `city` و`channel` و`customer_tier`. اعرض لكل منها القيم المميّزة وتكرار كل قيمة.

اثنان من الثلاثة نظيفان، وواحد ليس كذلك — والضرر غير مرئي حتى تنظر إلى النصوص بدقّة. عُدّ القيم
المميّزة في `city`، وكم مدينة فعلية تظنّها.

</div>

In [9]:
CATEGORICAL = ["channel", "customer_tier", "city"]

for col in CATEGORICAL:
    print(f"--- {col} ---")
    counts = raw[col].value_counts(dropna=False)
    for val, cnt in counts.items():
        print(f"  {val!r:20s} {cnt}")
    print()

print(f"city has {raw['city'].nunique()} distinct values")

--- channel ---
  'online'             2729
  'store'              1551
  'phone'              720

--- customer_tier ---
  'bronze'             2449
  'silver'             1785
  'gold'               766

--- city ---
  'Riyadh'             1214
  'Jeddah'             920
  'Dammam'             610
  'Tabuk'              419
  'Abha'               366
  'riyadh'             269
  'Riyadh '            264
  ' Jeddah'            192
  'JEDDAH'             188
  'abha'               141
  'DAMMAM'             133
  'dammam '            124
  'tabuk'              82
  'TABUK '             78

city has 14 distinct values


### Task 2.4 — Prove the date column is not one format

`order_date` is text. Your job is to show it holds **more than one** written format, and to say how
many of each.

Do not parse it yet. Classify the raw strings by their *shape* — a regular expression per format
you spot — and count how many rows match each. Print any row that matches none; that is how you
find the format you missed.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — أثبت أن عمود التاريخ ليس بصيغة واحدة

`order_date` نص. ومهمتك أن تُظهر أنه يحمل **أكثر من صيغة** مكتوبة، وأن تذكر عدد كل صيغة.

لا تحلّله إلى تاريخ بعد. صنّف النصوص الخام حسب **شكلها** — تعبير نمطي لكل صيغة تلاحظها — وعُدّ
الصفوف المطابقة لكل منها. واطبع أي صف لا يطابق أي صيغة؛ هكذا تجد الصيغة التي فاتتك.

</div>

In [18]:
print(raw["order_date"].sample(15, random_state=0).tolist())

patterns = {
    "iso_dash":       r"^\d{4}-\d{2}-\d{2}$",           # 2024-01-20
    "yyyy_slash":     r"^\d{4}/\d{2}/\d{2}$",           # 2024/12/07
    "mm_dd_slash":    r"^\d{2}/\d{2}/\d{4}$",           # 06/18/2024
    "day_mon_year":   r"^\d{1,2} [A-Za-z]{3} \d{4}$",   # 15 Sep 2024
}

shape_counts = {}
matched_any = pd.Series(False, index=raw.index)

for name, pattern in patterns.items():
    mask = raw["order_date"].astype(str).str.match(pattern)
    shape_counts[name] = int(mask.sum())
    matched_any |= mask

for name, count in shape_counts.items():
    print(f"    {name:12s} {count:5d} rows")
print(f"\n    matching none: {int((~matched_any).sum())}")

if (~matched_any).sum() > 0:
    print("\nunmatched examples:")
    print(raw.loc[~matched_any, "order_date"].head(10).tolist())

['2024-01-20', '2024-06-03', '2024/12/07', '06/18/2024', '02/01/2025', '2025/01/18', '2024/09/21', '04/07/2024', '06/08/2025', '2024/06/24', '03/21/2025', '03/26/2025', '01/09/2025', '1 Feb 2025', '23 May 2025']
    iso_dash      1317 rows
    yyyy_slash    1204 rows
    mm_dd_slash   1225 rows
    day_mon_year  1254 rows

    matching none: 0


### Task 2.5 — Separate the impossible from the merely unusual

Run `describe()` on the numeric columns and read it properly: for each column ask what the minimum
and the maximum would mean in the real shop.

Then find the rows that are **impossible** rather than surprising. A large order is surprising. An
order that shipped before it was placed is impossible, and no amount of domain knowledge makes it
fine. Count how many rows break at least one rule you can justify.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — افصل المستحيل عن الغريب فحسب

شغّل `describe()` على الأعمدة الرقمية واقرأها كما ينبغي: اسأل عن كل عمود ماذا تعني قيمته الدنيا
والعليا في المتجر الحقيقي.

ثم جِد الصفوف **المستحيلة** لا المفاجئة. فالطلب الكبير مفاجئ، أما الطلب الذي شُحن قبل أن يُقدَّم
فمستحيل، ولا تجعله أي معرفة بالمجال مقبولًا. عُدّ الصفوف التي تخرق قاعدة واحدة على الأقل تستطيع
تبريرها.

</div>

In [11]:
print(raw.describe().to_string(float_format="{:.2f}".format))
print()

order_date_parsed = pd.to_datetime(raw["order_date"], format="mixed", errors="coerce")
ship_date_parsed = pd.to_datetime(raw["ship_date"], format="mixed", errors="coerce")

rule_breaks = {
    "shipped_before_ordered": ship_date_parsed < order_date_parsed,
    "non_positive_quantity": raw["quantity"] <= 0,
    "non_positive_price": raw["unit_price"] <= 0,
    "future_order_date": order_date_parsed > pd.Timestamp.now(),
}

for name, mask in rule_breaks.items():
    print(f"    {name:26s} {int(mask.sum()):5d} rows")

       order_id  quantity  unit_price  discount_pct  commission_paid  refund_amount  revenue  returned
count   5000.00   5000.00     5000.00       3527.00          5000.00        5000.00  5000.00   5000.00
mean  102445.32     20.86       47.71          0.15            24.70         115.77   823.46      0.23
std     1408.71     11.65       37.46          0.08            26.10         358.73   870.04      0.42
min   100001.00      1.00        3.09          0.00             0.14           0.00     4.50      0.00
25%   101229.75     11.00       23.57          0.09             7.73           0.00   257.61      0.00
50%   102450.50     21.00       37.59          0.14            16.91           0.00   563.69      0.00
75%   103662.25     31.00       59.11          0.20            32.34           0.00  1077.94      0.00
max   104880.00     40.00      375.87          0.45           360.03        4821.61 12000.89      1.00

    shipped_before_ordered         0 rows
    non_positive_quantity     

### Task 2.6 — Meet the target

Now look at `returned`, the column the rest of the week predicts. Report how many orders were
returned, how many were kept, and the **positive rate** — the share of orders that came back.

Then compute the score a model would get by always predicting "kept". Write that number down. It
is the number every model you build this week has to beat, and in D5 you will find out that
beating it is not the same as being useful.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — تعرّف على الهدف

انظر الآن إلى `returned`، العمود الذي تتنبّأ به بقية الأسبوع. اذكر كم طلبًا أُرجع، وكم طلبًا احتُفظ
به، و**النسبة الإيجابية** — أي حصة الطلبات التي عادت.

ثم احسب الدقّة التي سيحقّقها نموذج يتنبّأ دائمًا بـ«لم يُرجَع». دوّن هذا الرقم. فهو الرقم الذي على
كل نموذج تبنيه هذا الأسبوع أن يتجاوزه، وستكتشف في اليوم الخامس أن تجاوزه ليس مرادفًا للفائدة.

</div>

In [12]:
class_counts = raw["returned"].value_counts()
return_rate = raw["returned"].mean()
majority_accuracy = raw["returned"].value_counts(normalize=True).max()

print(f"kept     (0): {class_counts[0]:,}")
print(f"returned (1): {class_counts[1]:,}")
print(f"\npositive rate: {return_rate:.1%}")
print(f"always predicting 'kept' would be {majority_accuracy:.1%} accurate")

kept     (0): 3,846
returned (1): 1,154

positive rate: 23.1%
always predicting 'kept' would be 76.9% accurate


### Task 2.7 — Your own investigation

You have a target and a set of columns. Form **one** hypothesis about what makes a return more
likely, and test it with a `groupby`.

Pick something you can defend from how shops actually work — for example, that orders which took
longer to ship come back more often, or that a particular channel is worse than the others. State
the hypothesis in the markdown cell, test it in the code cell, and then say whether the data
supported it.

You are not proving causation here and you are not building a model. You are practising the loop:
*question → measurement → interpretation.*

<div dir="rtl" align="right">

### المهمة ٢٫٧ — تحقيقك الخاص

لديك هدف ومجموعة أعمدة. كوّن **فرضية واحدة** عمّا يجعل الإرجاع أرجح، واختبرها بـ`groupby`.

اختر شيئًا تستطيع الدفاع عنه من واقع عمل المتاجر — مثلًا أن الطلبات التي طال شحنها تعود أكثر، أو
أن قناة بعينها أسوأ من غيرها. اذكر الفرضية في خلية Markdown، واختبرها في خلية الشيفرة، ثم قل هل
أيّدتها البيانات.

أنت لا تثبت سببية هنا ولا تبني نموذجًا، بل تتدرّب على الحلقة: **سؤال ← قياس ← تفسير.**

</div>

**My hypothesis:** *Orders that took longer to ship are more likely to be returned, because slow delivery increases customer frustration and the chance the item arrives too late to be useful.

<div dir="rtl" align="right">

**فرضيتي:** *فرضيتي: الطلبات التي استغرق شحنها وقتًا أطول تكون أكثر عرضة للإرجاع، لأن التأخير يزيد من استياء العميل وقد يجعل وصول المنتج متأخرًا عن الحاجة إليه.*

</div>

In [20]:
# ────────────────────────────────────────────────────────────────────
# 1) Turn your hypothesis into a column you can group by. If it is about shipping
#    speed, you need the number of days between the two dates first.
# 2) A continuous column has to be bucketed before it can be grouped — cut it into
#    a handful of bins.
# 3) Group by that column and take the mean of `returned`. That mean IS the return
#    rate of the group, which is why the 0/1 encoding pays off.
# 4) Print the group sizes next to the rates — a rate over 12 rows is not evidence.
# Search: "pandas groupby agg mean count"
# ────────────────────────────────────────────────────────────────────

days_to_ship = (ship_date_parsed - order_date_parsed).dt.days
raw["ship_speed_bucket"] = pd.cut(days_to_ship, bins=[-1, 1, 3, 7, 14, 100],
                                    labels=["0-1d", "2-3d", "4-7d", "8-14d", "15d+"])

finding = raw.groupby("ship_speed_bucket").agg(
    return_rate=("returned", "mean"),
    n_rows=("returned", "count")
)

print(finding.to_string(formatters={"return_rate": "{:.1%}".format}))
raw = raw.drop(columns=["ship_speed_bucket"])
print(raw.shape)   # لازم يطلع (5000, 13)

                  return_rate  n_rows
ship_speed_bucket                    
0-1d                    21.4%     336
2-3d                    21.6%     672
4-7d                    22.4%    1474
8-14d                   24.1%    2518
(5000, 13)


What I found: The data mildly supported my hypothesis — the return rate rises slightly as shipping takes longer, from 21.4% for 0-1 day shipments up to 24.1% for 8-14 day shipments. The effect is real but small (about 2.7 percentage points across the range), and group sizes are all reasonably large (336 to 2,518 rows), so this isn't noise from a tiny sample. However, the trend is gradual rather than a sharp jump, so shipping speed alone is a weak predictor on its own

<div dir="rtl" align="right">

ما وجدته: أيّدت البيانات فرضيتي بشكل طفيف — نسبة الإرجاع ترتفع قليلًا كلما طال وقت الشحن، من 21.4% للشحنات خلال يوم أو يومين إلى 24.1% للشحنات التي استغرقت 8-14 يومًا. الأثر حقيقي لكنه صغير (فرق حوالي 2.7 نقطة مئوية)، وأحجام المجموعات كلها كبيرة بما يكفي (من 336 إلى 2,518 صفًا) فهو مو ضجيج ناتج من عيّنة صغيرة. لكن الاتجاه تدريجي وليس قفزة حادة، فسرعة الشحن وحدها مؤشر ضعيف نسبيًا

</div>

## Section 3 — Stretch  (≈30 min)

Open-ended, and lower expectation of completeness.

Two of the columns in this file could not have been known at the moment the order was placed. Find
them, and for each one write a sentence explaining *when* its value becomes available.

You are not being asked to prove they are dangerous — that is D3's job, and it has a name for what
they are. You are being asked to practise the question that finds them: **"at the instant I would
need to make this prediction, would I actually have this number?"**

**Link to your capstone:** the first thing to do with your own dataset is exactly this. Before you
model anything, list every column and write down when its value becomes known. Columns that arrive
after the thing you are predicting are the single most common reason a capstone model scores
beautifully and fails in the demo.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل.

هناك عمودان في هذا الملف لم يكن من الممكن معرفتهما لحظة تقديم الطلب. جِدهما، واكتب لكل منهما جملة
تشرح **متى** تصبح قيمته متاحة.

لا يُطلب منك إثبات خطورتهما — فتلك مهمة اليوم الثالث، وله اسم لما هما عليه. بل يُطلب منك التدرّب
على السؤال الذي يجدهما: **«في اللحظة التي أحتاج فيها إلى هذا التنبّؤ، هل سيكون هذا الرقم بحوزتي
فعلًا؟»**

**الصلة بمشروعك:** أول ما تفعله بمجموعة بياناتك هو هذا بالضبط. قبل أن تبني أي نموذج، اسرد كل عمود
ودوّن متى تصبح قيمته معروفة. فالأعمدة التي تصل بعد الشيء الذي تتنبّأ به هي السبب الأول لأن يحقّق
مشروع التخرّج نتيجة رائعة ثم يفشل في العرض.

</div>

In [15]:
print(raw.columns.tolist())

suspicious_cols = ["commission_paid", "refund_amount"]

for col in suspicious_cols:
    print(f"\n--- {col} by returned ---")
    print(raw.groupby("returned")[col].describe())

['order_id', 'order_date', 'ship_date', 'city', 'channel', 'customer_tier', 'quantity', 'unit_price', 'discount_pct', 'commission_paid', 'refund_amount', 'revenue', 'returned', 'ship_speed_bucket']

--- commission_paid by returned ---
           count       mean        std   min      25%     50%      75%     max
returned                                                                      
0         3846.0  23.159974  25.067252  0.14   7.1850  15.810  30.0775  360.03
1         1154.0  29.849471  28.702383  0.24  10.3375  21.965  39.9075  253.40

--- refund_amount by returned ---
           count        mean         std  min       25%     50%       75%      max
returned                                                                          
0         3846.0    0.020023    0.909563  0.0    0.0000    0.00    0.0000    54.91
1         1154.0  501.539324  603.581519  0.0  111.3425  286.61  647.6225  4821.61


1. commission_paid — becomes known after the revenue is recognised (a flat 3% of revenue, computed once the sale is finalized, not at order time)
2. refund_amount — becomes known only after a return is accepted, so it doesn't exist at all until returned=1 actually happens

<div dir="rtl" align="right">

١. commission_paid — تصبح معروفة بعد الاعتراف بالإيراد (نسبة ثابتة 3% من revenue، تُحسب بعد إتمام البيع فعليًا، لا وقت تقديم الطلب)
٢. refund_amount — لا تصبح معروفة إلا بعد قبول عملية الإرجاع، فهي غير موجودة أصلًا إلا بعد وقوع returned=1 فعليًا
</div>

## Save your artefact

`data_quality_notes.md` is the handover to D3. It records what you measured today, with the actual
numbers, so tomorrow's cleaning is driven by evidence instead of memory. D2 reads it too, as the
list of things worth plotting.

<div dir="rtl" align="right">

## احفظ مخرجاتك

ملف `data_quality_notes.md` هو التسليم إلى اليوم الثالث. يسجّل ما قِسته اليوم بالأرقام الفعلية،
فيقود التنظيف غدًا بالأدلّة لا بالذاكرة. ويقرؤه اليوم الثاني أيضًا بوصفه قائمة بما يستحق الرسم.

</div>

In [21]:
report_md = f"""# Data Quality Notes — messy_sales

## Missing values
{missing.to_string(formatters={"rate": "{:.1f}%".format})}

`discount_pct` missing rate by channel:
{by_channel.to_string(float_format="{:.1%}".format)}

## Duplicates
{n_duplicates} exact duplicate rows.
{comparison.to_string(float_format="{:.4f}".format)}

## Category spellings
`city` has {raw['city'].nunique()} distinct values (more than the real number of cities).

## Date formats
{chr(10).join(f"- {name}: {count} rows" for name, count in shape_counts.items())}
- matching none: {int((~matched_any).sum())}

## Impossible rows
{chr(10).join(f"- {name}: {int(mask.sum())} rows" for name, mask in rule_breaks.items())}

## Target balance
Return rate: {return_rate:.1%}
Majority-class accuracy: {majority_accuracy:.1%}

## Leaked columns
- commission_paid: flat 3% of revenue, known only after revenue is recognised
- refund_amount: known only after a return is accepted
"""

out = ARTEFACT_DIR / "data_quality_notes.md"
out.write_text(report_md, encoding="utf-8")

print(f"Saved {out}\n")
print(out.read_text(encoding="utf-8"))

Saved c:\AIEP_Olo_student\week_2_data_engineering\labs_v2\D1_understand_the_data\artefacts\data_quality_notes.md

# Data Quality Notes — messy_sales

## Missing values
              missing  rate
discount_pct     1473 29.5%

`discount_pct` missing rate by channel:
channel
online    17.5%
phone    100.0%
store     17.7%

## Duplicates
120 exact duplicate rows.
                 rows  mean_revenue  mean_returned
with_duplicates  5000      823.4627         0.2308
deduplicated     4880      823.0218         0.2299

## Category spellings
`city` has 14 distinct values (more than the real number of cities).

## Date formats
- iso_dash: 1317 rows
- yyyy_slash: 1204 rows
- mm_dd_slash: 1225 rows
- day_mon_year: 1254 rows
- matching none: 0

## Impossible rows
- shipped_before_ordered: 0 rows
- non_positive_quantity: 0 rows
- non_positive_price: 0 rows
- future_order_date: 0 rows

## Target balance
Return rate: 23.1%
Majority-class accuracy: 76.9%

## Leaked columns
- commission_paid: flat 3% of 

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

</div>

In [22]:
# --- Sanity checks ----------------------------------------------------------------

check(len(raw) == 5000 and raw.shape[1] == 13,
      f"messy_sales should load as 5000 x 13, got {raw.shape[0]} x {raw.shape[1]}",
      f"يجب أن تُحمّل messy_sales بحجم ٥٠٠٠ × ١٣، والقيمة الحالية {raw.shape[0]} × {raw.shape[1]}")

check(0.20 < return_rate < 0.26,
      f"the returned rate should be about 23%, you measured {return_rate:.1%}",
      f"يجب أن تكون نسبة الإرجاع نحو ٢٣٪، وقد قِست {return_rate:.1%}")

check(n_duplicates > 0,
      f"there are duplicate rows in this file — you counted {n_duplicates}",
      f"يوجد صفوف مكرّرة في هذا الملف — وقد عددت {n_duplicates}")

check(raw["city"].nunique() > 5,
      f"city should have more spellings than real cities; you found {raw['city'].nunique()}",
      f"يجب أن تزيد تهجئات city عن عدد المدن الحقيقية، وقد وجدت {raw['city'].nunique()}")

check(sum(1 for c in shape_counts.values() if c > 0) >= 3,
      f"order_date should hold at least 3 written formats, you classified "
      f"{sum(1 for c in shape_counts.values() if c > 0)}",
      f"يجب أن يحمل order_date ثلاث صيغ مكتوبة على الأقل، وقد صنّفت "
      f"{sum(1 for c in shape_counts.values() if c > 0)}")

check(missing.loc["discount_pct", "rate"] > 10,
      f"discount_pct should be more than 10% missing, you measured "
      f"{missing.loc['discount_pct', 'rate']:.1f}%",
      f"يجب أن يتجاوز النقص في discount_pct ١٠٪، وقد قِست "
      f"{missing.loc['discount_pct', 'rate']:.1f}%")

check((ARTEFACT_DIR / "data_quality_notes.md").exists(),
      "data_quality_notes.md should exist in artefacts/ — D3 reads it tomorrow",
      "يجب أن يوجد data_quality_notes.md في artefacts/ — يقرؤه اليوم الثالث غدًا")

report()

──────────────────────────────────────────────────────────────────
  ✓  messy_sales should load as 5000 x 13, got 5000 x 13
  ✓  the returned rate should be about 23%, you measured 23.1%
  ✓  there are duplicate rows in this file — you counted 120
  ✓  city should have more spellings than real cities; you found 14
  ✓  order_date should hold at least 3 written formats, you classified 4
  ✓  discount_pct should be more than 10% missing, you measured 29.5%
  ✓  data_quality_notes.md should exist in artefacts/ — D3 reads it tomorrow
──────────────────────────────────────────────────────────────────
  ✅ All 7 checks passed. / اجتزت جميع الفحوصات (7).
──────────────────────────────────────────────────────────────────


## What's next

Tomorrow (D2) you take the same file and the same target and **look** at it: the class balance as a
chart, the return rate broken down by every categorical column you catalogued today, and the
relationships between the numeric ones. Today you found out what is wrong with the data; tomorrow
you find out what it has to say.

<div dir="rtl" align="right">

## ماذا بعد

غدًا (اليوم الثاني) تأخذ الملف نفسه والهدف نفسه و**تنظر** إليه: توازن الفئات كرسم بياني، ونسبة
الإرجاع موزّعة حسب كل عمود فئوي فهرسته اليوم، والعلاقات بين الأعمدة الرقمية. اليوم عرفت ما الخطأ
في البيانات، وغدًا تعرف ما الذي تقوله.

</div>